# RAG 프로젝트 최종 정리 - 2026-05-31

## 결론

최종적으로는 **KURE embedding + Chroma 690 index + dense query decomposition/RRF + document scoring + target-aware retrieval**을 검색 기본값으로 두고, 생성 쪽은 **Qwen3 8B 4bit + PEFT v2 adapter + source_store/sidecar 기반 context + 서비스용 history 보정**까지 연결했다.

검색 자체는 Phase 3/4 gold 50문항 기준으로 `hit@5=0.9633`, `nDCG@5=0.9215`, `multi_doc_recall@5=0.9127`까지 올라갔다. 다만 생성 평가는 아직 낮다. 가장 큰 이유는 **정답 문서는 찾는데, 실제 답변에 필요한 evidence chunk나 source_store final 값이 gold와 충돌하는 경우가 남아 있기 때문**이다. 그래서 PEFT보다 먼저 retrieval/evidence 품질과 context 조립을 많이 만졌고, 최종 서비스는 간단한 채팅 UX와 히스토리 처리까지 가능한 상태로 정리했다.

## 1. Retrieval 최종 결과

| variant | 문항수 | hit@5 | MRR@5 | nDCG@5 | doc_recall@5 | multi_doc_recall@5 |
| --- | --- | --- | --- | --- | --- | --- |
| 104_best_dense_qdecomp_docscore_targetaware | 50 | 0.9633 | 0.9300 | 0.9215 | 0.9633 | 0.9127 |
| 109_best_target50 | 50 | 0.9633 | 0.9300 | 0.9215 | 0.9633 | 0.9127 |
| 108_best_preserve5 | 50 | 0.9567 | 0.9300 | 0.9175 | 0.9567 | 0.8968 |
| 107_hybrid_docscore_targetaware | 50 | 0.9333 | 0.9400 | 0.9144 | 0.9333 | 0.8413 |
| 105_dense_keyword_rerank | 50 | 0.8767 | 0.9500 | 0.8710 | 0.8767 | 0.7063 |
| 106_hybrid_keyword_rerank | 50 | 0.8167 | 0.9083 | 0.8221 | 0.8167 | 0.5635 |

핵심 변화는 다음과 같다.

- 단순 dense보다 query decomposition + RRF를 붙여 질문이 여러 조건을 갖는 경우 후보 문서를 더 넓게 확보했다.
- chunk top-5를 그대로 쓰지 않고, 문서 단위 doc scoring과 target-aware 보정을 넣었다.
- 같은 문서 chunk가 top-k를 독식하지 않도록 document-level selection을 적용했다.
- hybrid/rerank도 비교했지만, 이 데이터에서는 hybrid/keyword rerank가 target 문서와 무관한 키워드 chunk를 끌어올리는 경우가 있어 최종 후보에서는 dense 계열이 더 안정적이었다.

## 2. Evidence Recall 진단과 개선

| 설정 | evidence_recall@5 | evidence_hit@5 | context_evidence_recall | doc_recall@5 | doc hit but evidence missed | 해석 |
| --- | --- | --- | --- | --- | --- | --- |
| 초기 104 remap 진단 | 0.0200 |  |  | 0.8400 | 39 | 정답 문서는 찾지만 근거 chunk가 거의 빠지는 상태 |
| baseline_existing_context | 0.2096 | 0.6600 | 0.2146 | 0.8033 | 17 | 기존 context 기준 |
| existing_core_rr_source_final | 0.3496 | 0.8800 | 0.6849 | 0.7433 | 3 | 가장 안정적인 generation 연결 후보 |
| existing_canonical_source_final | 0.3705 | 0.8000 | 0.8642 | 0.7433 | 7 | context 전체 근거 포함률은 높지만 문서별 안정성은 낮음 |
| strict_pack required-first adaptive | 0.3821 | 0.8800 | 0.8875 | 0.7433 | 3 | 최종 sweep에서 균형이 좋았던 후보 |

이 과정에서 가장 중요한 발견은 `정답 문서 검색 성공`과 `정답 근거 검색 성공`이 다르다는 점이었다. 초기에 104 retrieval은 문서 recall은 높았지만 evidence recall은 매우 낮았다. 이후 source_store/fact_type 기반 evidence selection, required fact 우선, adaptive quota를 적용해 **문서는 맞았는데 근거 chunk가 빠지는 문제를 17건에서 3건 수준으로 줄였다.**

## 3. Generation / Context 실험

| variant | Phase3 task | budget | required field | unanswerable | hit@5 | nDCG@5 | multi_doc_recall@5 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| previous_final_target_evidence_50 | 0.3097 | 0.2083 | 0.1656 | 0.5500 | 0.9000 | 0.8741 | 0.7619 |
| new_preserve_top_50 | 0.3033 | 0.2083 | 0.1111 | 0.6000 | 0.9633 | 0.9215 | 0.9127 |

생성에서는 다음 시행착오가 있었다.

- source_store를 많이 넣으면 정보는 늘지만 noisy해졌다.
- raw retrieval top 문서를 전부 보존하면 retrieval metric은 좋아졌지만 generation Phase3 점수는 오르지 않았다.
- 예산 질문에서는 금액 정규화가 필요했다. 예: `1,515,000천원`은 원문값과 `1,515,000,000원` 환산값을 같이 넘기도록 보강했다.
- source_store의 `final_budget`, `final_period`는 유용하지만, gold 또는 원문 chunk와 충돌하는 경우가 있어 **retrieved chunk에서도 확인될 때 더 신뢰**하도록 처리했다.
- 질문 유형별 템플릿을 적용했다. 예산은 금액 중심, 제출서류/자격요건은 항목 중심, 다중 문서는 문서별 분리 답변을 유도했다.

최종 OpenAI Judge 평가 요약은 아래와 같다.

| 항목 | 값 |
| --- | --- |
| Phase3 evaluated_count | 50 |
| Phase3 task score mean | 0.30966666666666665 |
| Budget numeric accuracy | 0.20833333333333334 |
| Required field accuracy | 0.16555555555555557 |
| Unanswerable refusal accuracy | 0.55 |
| Phase4 mode/model | api / gpt-4o-mini |
| Phase4 reference mode | evidence_only |
| Phase4 calculated overall | 3.7820000000000005 |
| Phase4 judged count | 50 |
| Phase4 needs human review | 23 |

주의할 점은 Phase 4가 `evidence_only` 기준이라는 것이다. 즉, 모델 답변이 제공된 evidence에 잘 근거했는지를 보는 보조 지표라서, gold 정답과의 엄격한 일치 여부는 Phase 3를 같이 봐야 한다. 실제로 Q003처럼 evidence/source_store에는 `2,000,000,000원`이 있지만 gold는 다른 금액을 요구하는 케이스가 있어 Phase4는 높고 Phase3는 낮게 나올 수 있다.

## 4. PEFT 실험

| 항목 | 결과 | 해석 |
| --- | --- | --- |
| Base Qwen valid loss | 1.5579 | 비교 기준 |
| PEFT v2 adapter valid loss | 0.7741 | 약 50.3% loss 개선 |
| v5 추가 정답형 라벨 | 13개 | 총 63개까지 늘렸지만 추가 학습 신호는 아직 작음 |
| 현재 서비스 adapter | qwen3_8b_qlora_sft_v2_truncfix/adapter | 가장 안정적인 현재 후보 |

PEFT는 파이프라인 자체는 성공했다. v2 adapter는 base 대비 validation loss를 약 50% 줄였고, 답변 스타일도 gold 형식에 가까워졌다. 하지만 50개 라벨만으로는 사실성을 안정화하기 어렵고, 추가 라벨 확장 과정에서 답변불가/부분답변 비중이 커져 v3/v4/v5는 최종 채택하지 않았다. 그래서 현재는 **v2 adapter를 서비스 옵션/기본 후보로 유지**하고, 정답형 라벨 50~100개를 추가 확보한 뒤 PEFT를 재개하는 게 맞다.

## 5. 서비스 상태

현재 서비스는 `scripts/rag_service_web.py`에 정리되어 있다.

- 기본 chunk: `data/processed/chunks_v2_690.jsonl`
- 기본 index: `indexes/chroma_kure_v1_chunks_v2_690/`
- 기본 source_store: `data/processed/source_store_v2_690.jsonl`
- 기본 모델: `unsloth/Qwen3-8B-bnb-4bit`
- 기본 adapter: `outputs/peft/qwen3_8b_qlora_sft_v2_truncfix/adapter`
- 실행 예시: `.venv/bin/python scripts/rag_service_web.py --host 127.0.0.1 --port 7860 --use-best-adapter`

서비스 UX 쪽에서는 실험용 패널을 걷어내고, ChatGPT처럼 질문/답변만 보이도록 바꿨다. Enter 전송, Shift+Enter 줄바꿈, 첫 응답 모델 로딩 안내, 생성 중 표시도 넣었다. 그리고 히스토리에서는 이전 질문 전체를 그대로 검색어로 쓰지 않고 `focus_subject`, `recent_amount`, `recent_docs`를 추출해 후속 질문을 처리하도록 바꿨다.

예를 들어 `고려대 사업 예산 알려줘` 이후 `그러면 그거 어디에 쓰기로했어?`를 물으면 검색을 다시 돌리지 않고 최근 답변의 대상 사업을 사용해 `그 예산은 고려대학교 차세대 포털·학사 정보시스템 구축 사업에 쓰이는 예산입니다.`처럼 짧게 답하도록 했다.

## 6. 실제 사례에서 본 변화

- 원래 문제: retrieval top 문서는 맞는데 정답 evidence가 context에 없어서 LLM이 엉뚱한 chunk의 금액/기간을 답했다.
- 적용한 해결: evidence recall 진단을 별도로 만들고, source_store/fact_type/required field 기반으로 문서별 핵심 evidence를 다시 고르게 했다.
- 결과: `doc_hit_but_evidence_missed`가 17건에서 3건으로 줄었다.
- 남은 문제: source_store의 final 값 자체가 gold와 충돌하거나, gold가 원문/파싱 데이터와 다르게 잡힌 경우가 있어, 단순히 source_store를 더 믿는 방식은 위험하다.

## 7. 중요한 파일

| 역할 | 경로 |
| --- | --- |
| 서비스 Web | scripts/rag_service_web.py |
| 서비스 CLI | scripts/rag_service_cli.py |
| 최신 690 chunks | data/processed/chunks_v2_690.jsonl |
| 최신 690 source store | data/processed/source_store_v2_690.jsonl |
| 최신 Chroma index | indexes/chroma_kure_v1_chunks_v2_690/ |
| 최종 retrieval sweep | outputs/eval/light_retrieval_sweep_690_phase34_gold50/summary_metrics.csv |
| 최종 OpenAI judge eval | outputs/eval/final_690_phase34_gold_qwen_openai_judge/ |
| 최종 generation predictions | outputs/generation/final_690_phase34_gold_qwen/126_service_route_v3_nonbudget_patch_123_budget_50_eval_predictions.jsonl |
| PEFT best adapter | outputs/peft/qwen3_8b_qlora_sft_v2_truncfix/adapter/ |
| 이 리포트 HTML | outputs/reports/final_rag_wrapup_20260531.html |

## 8. 앞으로 남은 개선

1. 정답형 PEFT 라벨을 50~100개 더 확보한 뒤 v2 adapter를 기준으로 재학습한다.
2. source_store final 값은 retrieved chunk에서 같은 값이 확인될 때만 최종값으로 쓰는 정책을 더 강하게 적용한다.
3. Phase3에서 낮게 나오는 budget/required_fields 유형을 우선 개선한다.
4. evidence recall은 기존 eval과 분리된 참고 지표로 유지하되, 실험마다 같이 뽑아 retrieval/context 병목을 빨리 찾는다.
5. 서비스는 현재 형태로 데모 가능하지만, 근거 문서 보기/출처 열람은 사용자용 옵션으로만 붙이는 게 좋다.



## 9. HTML 샘플 표 보강

HTML 리포트의 실제 문항 비교 표는 `eval/evaluation/data/rfp_domain_gold_sample.jsonl`의 gold 구조를 함께 읽어 생성 답변과 정답 기준을 나란히 보여준다. 판정은 Phase3 자동평가 기준이고, Phase4는 evidence-only judge라 보조 지표로만 해석해야 한다.
